In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

JupyterDash.infer_jupyter_proxy_config()

# Configure the necessary Python module imports for dashboard components
import os
import base64
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

# Import enhanced CRUD module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Credentials can be read from environment variables for better security.
# The default values keep the project runnable in the original Codio classroom environment.
username = os.getenv("AAC_USERNAME", "aacuser")
password = os.getenv("AAC_PASSWORD", "SNHU1234")

db = AnimalShelter(username, password)


def load_data(query=None):
    """Read shelter data into a DataFrame and safely remove MongoDB's _id column."""
    records = db.read(query or {})
    dff = pd.DataFrame.from_records(records)
    if "_id" in dff.columns:
        dff = dff.drop(columns=["_id"])
    return dff


# Start with all records for the initial dashboard view.
df = load_data()

#########################
# Pre-build MongoDB queries for filters
#########################

RESCUE_QUERIES = {
    "water": {
        "$and": [
            {"animal_type": "Dog"},
            {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}},
            {"sex_upon_outcome": "Intact Female"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}},
        ]
    },
    "mountain": {
        "$and": [
            {"animal_type": "Dog"},
            {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}},
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}},
        ]
    },
    "disaster": {
        "$and": [
            {"animal_type": "Dog"},
            {"breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}},
            {"sex_upon_outcome": "Intact Male"},
            {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}},
        ]
    },
    "reset": {},
}


def build_columns(dff):
    """Build table columns dynamically from the current DataFrame."""
    return [{"name": column, "id": column, "deletable": False, "selectable": True} for column in dff.columns]


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Logo handling now fails gracefully if the image file is unavailable.
image_filename = "Grazioso Salvare Logo.png"
if os.path.exists(image_filename):
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()
    logo_component = html.Img(src=f"data:image/png;base64,{encoded_image}", style={"height": "80px", "width": "auto"})
else:
    logo_component = html.Div("Grazioso Salvare", style={"fontWeight": "bold", "fontSize": "24px"})

app.layout = html.Div([
    html.Div(
        style={"display": "flex", "alignItems": "center", "justifyContent": "space-between"},
        children=[
            html.A(logo_component, href="https://www.snhu.edu", target="_blank"),
            html.Div([
                html.H1("Grazioso Salvare Dashboard", style={"margin-bottom": "0"}),
                html.H4("Created by Ahmad Mansour", style={"margin-top": "0"}),
            ], style={"textAlign": "right"}),
        ],
    ),

    html.Hr(),

    html.Div(id="filter-div", children=[
        html.H5("Filter by Rescue Type:"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset (All Records)", "value": "reset"},
            ],
            value="reset",
            labelStyle={"display": "inline-block", "margin-right": "15px"},
        ),
    ]),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=build_columns(df),
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "padding": "5px", "fontFamily": "Arial", "fontSize": 12},
        style_header={"fontWeight": "bold", "backgroundColor": "#f2f2f2"},
    ),

    html.Br(),
    html.Hr(),
    html.Div(id="graph-id", style={"width": "100%", "margin": "0 auto"}),
    html.Br(),
    html.Hr(),
    html.Div(id="map-id", style={"width": "100%", "margin": "0 auto"}),
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [Output("datatable-id", "data"), Output("datatable-id", "columns"), Output("datatable-id", "selected_rows")],
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    query = RESCUE_QUERIES.get(filter_type, {})
    dff = load_data(query)
    return dff.to_dict("records"), build_columns(dff), [0] if not dff.empty else []


@app.callback(Output("graph-id", "children"), [Input("datatable-id", "derived_virtual_data")])
def update_graphs(view_data):
    if not view_data:
        return html.Div("No records available for the selected filter.")

    dff = pd.DataFrame.from_dict(view_data)
    if "breed" not in dff.columns or dff.empty:
        return html.Div("Breed data is unavailable for the selected records.")

    fig = px.pie(dff, names="breed", title="Breed Distribution for Selected Dogs")
    return dcc.Graph(figure=fig)


@app.callback(Output("datatable-id", "style_data_conditional"), [Input("datatable-id", "selected_columns")])
def update_styles(selected_columns):
    return [{"if": {"column_id": column}, "background_color": "#D2F3FF"} for column in selected_columns or []]


@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"), Input("datatable-id", "derived_virtual_selected_rows")],
)
def update_map(view_data, selected_rows):
    if not view_data:
        return html.Div("No map data available for the selected filter.")

    dff = pd.DataFrame.from_dict(view_data)
    required_columns = {"location_lat", "location_long", "breed", "name"}
    if not required_columns.issubset(dff.columns):
        return html.Div("Location data is unavailable for the selected records.")

    row = selected_rows[0] if selected_rows else 0
    if row >= len(dff):
        row = 0

    selected_animal = dff.iloc[row]
    latitude = selected_animal["location_lat"]
    longitude = selected_animal["location_long"]

    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div("The selected animal does not have valid location coordinates.")

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(str(selected_animal["breed"])),
                    dl.Popup([html.H4("Animal Name"), html.P(str(selected_animal["name"]))]),
                ],
            ),
        ],
    )


# Run app in Jupyter
app.run_server(mode="external", debug=True)

